# Session 8 — Gradio Chat Demo

This is a deliberately small UI example. The goal is to show how quickly the OpenAI SDK plus streaming can become an interactive app.

## Learning Goals

- connect a model call to a simple UI
- reuse the streaming generator pattern from the previous notebook
- understand why Gradio is useful for quick demos

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_ORG_ID = os.getenv('OPENAI_ORG_ID')
OPENAI_PROJECT_ID = os.getenv('OPENAI_PROJECT_ID')

print('OpenAI key present:', bool(OPENAI_API_KEY))
print('OpenAI org ID present:', bool(OPENAI_ORG_ID))
print('OpenAI project ID present:', bool(OPENAI_PROJECT_ID))

OpenAI key present: True
OpenAI org ID present: True
OpenAI project ID present: True


In [2]:
def stream_chat_response(message, history):
    client = OpenAI(api_key=OPENAI_API_KEY, organization=OPENAI_ORG_ID, project=OPENAI_PROJECT_ID)
    messages = []

    for user_text, assistant_text in history:
        messages.append({"role": "user", "content": user_text})
        messages.append({"role": "assistant", "content": assistant_text})

    messages.append({"role": "user", "content": message})

    stream = client.responses.create(
        model='gpt-4o-mini',
        input=messages,
        stream=True,
    )

    partial = ''
    for event in stream:
        if event.type == 'response.output_text.delta':
            partial += event.delta
            yield partial

## Build the Demo

In [3]:
demo = gr.ChatInterface(
    fn=stream_chat_response,
    title='Session 8 Gradio Demo',
    description='A minimal streaming chat app using the OpenAI SDK.'
)

demo

Gradio Blocks instance: 29 backend functions
--------------------------------------------
fn_index=0
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x0000023F37BD38C0>
 outputs:
 |-<gradio.components.textbox.Textbox object at 0x0000023F37BD38C0>
 |-<gradio.components.state.State object at 0x0000023F37DBB250>
fn_index=1
 inputs:
 |-<gradio.components.state.State object at 0x0000023F37DBB250>
 |-<gradio.components.chatbot.Chatbot object at 0x0000023F37BD34D0>
 outputs:
 |-<gradio.components.chatbot.Chatbot object at 0x0000023F37BD34D0>
fn_index=2
 inputs:
 |-<gradio.components.state.State object at 0x0000023F37DBB250>
 |-<gradio.components.state.State object at 0x0000023F37DBB890>
 outputs:
 |-<gradio.components.state.State object at 0x0000023F37DBB390>
 |-<gradio.components.chatbot.Chatbot object at 0x0000023F37BD34D0>
fn_index=3
 inputs:
 |-<gradio.components.chatbot.Chatbot object at 0x0000023F37BD34D0>
 outputs:
 |-<gradio.components.state.State object at 0x0000023F37DBB890

To launch locally, run:

```python
demo.launch()
```

Gradio is excellent for quick demos. In the next step we move to Streamlit for the main RAG app build.

In [4]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
